# Questão 2 — Dados, classificadores e avaliação

**Estado:** base, CV aninhada dos quatro classificadores-base e voto majoritário dos quatro implementados. Os experimentos 30×10, os intervalos de confiança e Friedman/Nemenyi já foram concluídos; as curvas de aprendizagem agora têm scripts e resultados em `resultados/curvas_aprendizagem/`.

Rótulos da Questão 1: **Log1p + MinMaxScaler**, K=3, H=3. Os 57 atributos do CSV permanecem na escala original. Pré-processadores são ajustados apenas nos folds de treino.

In [ ]:
from pathlib import Path
import sys
caminho_atual = Path.cwd()
raiz_repo = caminho_atual if (caminho_atual / 'questao2').is_dir() else caminho_atual.parent
if not (raiz_repo / 'questao2').is_dir():
    raise RuntimeError('Execute o notebook a partir da raiz do repositório ou da pasta questao2')
if str(raiz_repo) not in sys.path:
    sys.path.insert(0, str(raiz_repo))
from questao2.src.dados import carregar_dados, validar_base
from questao2.src.validacao import conferir_folds, gerar_folds_externos, gerar_folds_internos
import numpy as np
print('Raiz do projeto:', raiz_repo)

In [ ]:
info = validar_base()
X, respostas, atributos = carregar_dados()
print('Dimensões:', X.shape)
for alvo, y in respostas.items():
    valores, contagens = np.unique(y, return_counts=True)
    print(alvo, dict(zip(valores.tolist(), contagens.tolist())))

In [ ]:
for nome_alvo, rotulos in respostas.items():
    resumo = conferir_folds(rotulos)
    repeticao, fold, treino, teste = next(gerar_folds_externos(rotulos))
    internos = list(gerar_folds_internos(rotulos[treino]))
    print(nome_alvo, resumo, '1º fold externo:',len(treino),'treino +',len(teste),'teste;',len(internos),'folds internos')

## Bayes gaussiano multivariado — implementação e ensaio

O primeiro classificador já está implementado em `src/modelos/bayes_gaussiano.py`. Usa covariância completa por classe estimada por máxima verossimilhança e regularização isotrópica para tornar as matrizes invertíveis. **A matriz regularizada é uma adaptação numérica, não a MLE pura.**

O pipeline `log1p → MinMax → Bayes gaussiano` é ajustado dentro de cada CV. A célula abaixo roda **somente um fold externo por alvo**, cada qual com cinco folds internos para selecionar a regularização. Serve para confirmar execução, não é a avaliação 30×10 exigida.

In [ ]:
from questao2.src.avaliacao_gaussiano import avaliar_gaussiano

for nome_alvo, y in respostas.items():
    folds, _ = avaliar_gaussiano(X, y, nome_alvo, repeticoes=1, max_folds=1)
    print(nome_alvo, 'F1 macro:', round(folds[0]['f_measure_macro'], 4),
          'erro:', round(folds[0]['taxa_erro'], 4),
          'lambda:', folds[0]['regularizacao_relativa'])

### Como executar o experimento completo

No terminal, a partir da **raiz do repositório**:

```bash
python -m questao2.scripts.rodar_gaussiano --todos-folds --repeticoes 30
```

Esta execução (30×10 folds, com validação interna 5-fold) é muito mais custosa que o teste do notebook. O pacote conserva o diagnóstico histórico **1×10** e os resultados definitivos **30×10** em `resultados/gaussiano/`. Os intervalos de confiança descritivos e Friedman/Nemenyi exploratórios já foram calculados e estão em `resultados/analise_estatistica/`. O comando acima serve apenas para **reproduzir** o experimento completo.

## Bayes por k-vizinhos — implementação e ensaio

`BayesKVizinhos` estima as probabilidades a posteriori por fração dos k vizinhos por classe e utiliza as distâncias Euclidiana, City-Block e Chebyshev. Ajusta **k e distância por 5 folds internos**, usando F1 macro. O Log1p e o MinMax são ajustados apenas nos dados de treinamento correspondentes.

A célula seguinte é apenas um **smoke test de 1 fold por alvo**. Os resultados definitivos **já foram obtidos**, com 30 repetições de 10 folds externos. Esta célula executa apenas um ensaio curto para verificar o classificador.

In [ ]:
from questao2.src.avaliacao_knn import avaliar_knn

for nome_alvo, y in respostas.items():
    folds, _ = avaliar_knn(X, y, nome_alvo, repeticoes=1, max_folds=1)
    r = folds[0]
    print(nome_alvo, "F1 macro:", round(r["f_measure_macro"], 4),
          "erro:", round(r["taxa_erro"], 4),
          "k:", r["n_vizinhos"], "distância:", r["distancia"])

### Avaliação completa do k-NN

No terminal, na raiz do repositório:

```bash
python -m questao2.scripts.rodar_knn --todos-folds --repeticoes 30
```

Os resultados de **1×10** armazenados em `resultados/knn` são preliminares. A avaliação final 30×10, as métricas, os ICs descritivos, Friedman/Nemenyi e as curvas de aprendizagem já foram produzidos e estão em `resultados/`. Os resultados legados 1×10 neste diretório continuam apenas diagnósticos históricos.

## Bayes por janela de Parzen — kernel gaussiano produto

`BayesParzen` implementa densidade condicional por mistura de gaussianas
univariadas multiplicadas nas 57 dimensões, com largura escalar comum `h`.
Usa priors empíricas e log-sum-exp para estabilidade numérica. A busca de `h`
(grade predefinida) é feita **apenas** nos 5 folds internos, sobre dados
Log1p+MinMax ajustados separadamente em cada treino interno.

A célula abaixo executa **somente 1 fold externo por alvo**, não os 30 × 10
necessários para o experimento final.


In [ ]:
from questao2.src.avaliacao_parzen import avaliar_parzen, VALORES_H
print("Grade h:", VALORES_H)
for nome_alvo, y in respostas.items():
    folds, _ = avaliar_parzen(X, y, nome_alvo, repeticoes=1, max_folds=1)
    r = folds[0]
    print(nome_alvo, "F1 macro:", round(r["f_measure_macro"], 4),
          "erro:", round(r["taxa_erro"], 4),
          "h:", r["largura_janela"])


### Avaliação completa do Bayes Parzen

Na raiz do repositório:

```bash
python -m questao2.scripts.rodar_parzen --todos-folds --repeticoes 30
```

Os diagnósticos históricos **1 × 10 folds** e os resultados definitivos **30 × 10 folds** estão em `resultados/parzen/`. Intervalos de confiança descritivos, comparação estatística e curvas de aprendizagem já foram concluídos e salvos em `resultados/`.


## Regressão logística — quarto classificador (etapa 2d)

A regressão logística utiliza **regularização L2** e o otimizador **LBFGS**, com modelo binário para a classe original ou multinomial (softmax) para os três clusters. O ajuste interno escolhe `C` em `(0.03, 0.3, 3.0, 30.0, 300.0)` e `class_weight` em `(None, 'balanced')` por **F1 macro**. Toda transformação é ajustada somente nos treinos internos/externos; o teste externo fica intacto até a avaliação.

A célula a seguir roda apenas **um fold externo por versão**. É diagnóstico, **não** a avaliação final de `30 × 10` requerida.

In [ ]:
from questao2.src.avaliacao_logistica import avaliar_logistica, VALORES_C, PESOS_CLASSES
print("Valores de C:", VALORES_C, "Pesos de classe:", PESOS_CLASSES)
for nome_alvo, y in respostas.items():
    folds, _ = avaliar_logistica(X, y, nome_alvo, repeticoes=1, max_folds=1)
    r = folds[0]
    print(nome_alvo, "F1 macro:", round(r["f_measure_macro"], 4),
          "erro:", round(r["taxa_erro"], 4), "C:", r["C"],
          "class_weight:", r["class_weight"])

### Executar o protocolo completo de regressão logística

Na raiz do repositório:

```bash
python -m questao2.scripts.rodar_logistica --todos-folds --repeticoes 30
```

Os arquivos em `resultados/logistica/` incluem tanto a repetição diagnóstica `1×10` quanto os resultados finais `30×10`. Os ICs descritivos, Friedman/Nemenyi exploratórios e curvas de aprendizagem estão concluídos em `resultados/`.

## Voto majoritário — quinto classificador (etapa 2e)

Combina **predições externas** dos quatro classificadores nos mesmos folds, com **um voto por classificador**. Em empate, utiliza a maior priori estimada apenas no **treino externo**; no empate das próprias prioris, escolhe o menor rótulo. Não há novos hiperparâmetros.

A célula abaixo usa os arquivos 1×10 já salvos, conferindo o alinhamento e mostrando as métricas. Não é o protocolo completo 30×10.

In [ ]:
from questao2.src.avaliacao_voto import avaliar_voto_de_csvs
for alvo in respostas:
    folds, predicoes = avaliar_voto_de_csvs(alvo, repeticoes=1, max_folds=10)
    print(alvo, "folds:", len(folds), "predições:", len(predicoes),
          "F1 macro médio:", round(np.mean([x["f_measure_macro"] for x in folds]),4),
          "erro médio:", round(np.mean([x["taxa_erro"] for x in folds]),4),
          "empates:", sum(x["empates_resolvidos_priori"] for x in folds))

### Executar 30×10 para o quinto classificador

Primeiro executar os quatro scripts de modelos-base com `--todos-folds --repeticoes 30`. Depois, na raiz do projeto:

```bash
python -m questao2.scripts.rodar_voto --todos-folds --repeticoes 30
```

O script recusa combinações de folds incompatíveis. **O protocolo 30×10, os ICs descritivos, Friedman/Nemenyi e as curvas de aprendizagem foram concluídos.**

## Etapa 3A — Execução e estatística de 30 × 10 folds

**Os cinco classificadores e as duas versões do alvo têm o experimento completo 30 × 10 armazenado e auditado.**
Os scripts `executar_estudo.py` e `analisar_resultados.py` permitem reproduzir ou auditar os resultados. O `executar_estudo.py` conserva o progresso por repetição, evitando treinamento repetido de arquivos já válidos.
Para evitar reprocessar horas de treinamento, `executar_estudo` salva cada repetição isoladamente e valida todas as predições por fold.

```bash
python -m questao2.scripts.status_estudo
python -m questao2.scripts.executar_estudo --repeticoes 30
python -m questao2.scripts.analisar_resultados --repeticoes 30
```
Os testes Friedman/Nemenyi exploratórios e os ICs descritivos já foram calculados sobre as 30 repetições concluídas.


In [ ]:
from pathlib import Path
import pandas as pd
pasta_analise = Path("questao2/resultados/analise_estatistica")
if not pasta_analise.exists():
    pasta_analise = Path("resultados/analise_estatistica")
for alvo in ("y_original", "y_double_kmeans"):
    caminho = pasta_analise / alvo / "resumo_metricas.csv"
    if caminho.exists():
        tabela = pd.read_csv(caminho)
        print(f"{alvo}: resultados estatísticos consolidados 30x10 disponíveis")
        display(tabela.pivot(index="modelo", columns="metrica", values="media"))
    else:
        print(f"{alvo}: análise ainda não executada")


## Etapa 4 — Curvas de aprendizagem (Questão 2d)

As curvas utilizam proporções de 5% a 95% do conjunto de dados, com amostragem
estratificada. Para cada ponto, os cinco classificadores compartilham a mesma
divisão; os modelos-base escolhem hiperparâmetros via 5-fold *dentro do treino*.

As métricas de treino são in-sample; F1 de teste é obtido somente no conjunto
complementar. **Não reutilizar as predições da CV 30×10 para estas curvas.**

As 19 proporções já foram executadas para as duas variáveis resposta. O bloco seguinte abre os resultados salvos. Para apenas auditar/consolidar sem novo treino, execute `python -m questao2.scripts.rodar_curvas --repeticoes 1 --apenas-consolidar` na raiz do repositório.


In [ ]:
from pathlib import Path
import pandas as pd
p_curvas = Path('questao2/resultados/curvas_aprendizagem')
if not p_curvas.is_dir():
    p_curvas = Path('resultados/curvas_aprendizagem')
resumo = p_curvas / 'resumo_curvas.csv'
if resumo.exists():
    display(pd.read_csv(resumo).head(12))
else:
    print('Curvas ainda incompletas. Execute python -m questao2.scripts.rodar_curvas --repeticoes 1')

In [ ]:
from IPython.display import Image, display
for alvo in ('y_original', 'y_double_kmeans'):
    for conjunto in ('treinamento','teste'):
        img = p_curvas / 'graficos' / f'{alvo}_f1_{conjunto}.png'
        if img.exists():
            display(Image(filename=str(img)))